### ¿Qué es Apache Spark?

Apache spark es un framework de programación para el procesamiento de datos distribuidos (computacion en cluster) open source. Spark fue originalmente desarrolado por la Universidad de California Berkeley quien posteriomente donó el codigo del proyecto a la Apache Software Foundation, quien desde entonces se encarga de su mantenimiento.

Spark proporciona APIs para ser empleado en conjunto con lenguajes de programación como Java, Python, Scala y R. Además, cuenta con herramientas para la realización de tareas de alto nivel como Spark SQL (para el procesamiento de datos estructurados basado en SQL), MLlib (para la implementación de modelos de aprendizaje máquina), GraphX (para el procesamiento de grafos), etc.

### ¿Qué es PySpark?

PySpark es una herramienta que nos deja usar Spark encima de Python. Permitiendonos combinar el proceso de datos distribuidos de Spark con la simplicidad de Python para el análisis de conjuntos masivos de datos (big data).

In [1]:
!pip install pyspark
!pip install -U -q PyDrive
# !apt install openjdk-8-jdk-headless -qq

You should consider upgrading via the '/home/vania/jupyter/jupyterenv/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/home/vania/jupyter/jupyterenv/bin/python -m pip install --upgrade pip' command.


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

In [3]:
import pyspark
# Carga ufnciones extra
from pyspark.sql.functions import * 
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('primeros_pasos').getOrCreate()

22/02/12 10:57:37 WARN Utils: Your hostname, vania-Latitude-7400 resolves to a loopback address: 127.0.1.1; using 192.168.0.16 instead (on interface wlo1)
22/02/12 10:57:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
22/02/12 10:57:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### RDDs

La forma más elemental de trabajar con datos en PySpark es haciendo uso de un RDD (Resilient Distributed Dataset), el cual nos da la capacidad de manejar los datos y realizar cálculos con ellos de forma distribuida en varios nodos de un cluster.

Para realizar los ejemplos de este notebook haremos uso de una base de datos que cuenta con datos de los jugadores del juego FIFA20, dichos datos fueron tomados este concurso de kaggle donde podrán encontrar una descripción detalalda de cada campo de la base datos. https://www.kaggle.com/stefanoleone992/fifa-20-complete-player-dataset/version/1?select=players_20.csv

Podemos crear un RDD a partir de un archivo CSV de la siguiente manera

In [4]:
raw_data = spark.sparkContext.textFile('../data/players_20.csv')

In [5]:
raw_data

../data/players_20.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0

Una de las desventajas de los RDDs es que solo proveen a los datos de la estructura minima para poder trabajar con ellos. Imprimamos los primeros dos elementos del RDD para ver en que condicion se encuentran nuestros datos

In [6]:
for row in raw_data.take(2):
    print(row)
    print('\n')

sofifa_id,player_url,short_name,long_name,age,dob,height_cm,weight_kg,nationality,club,overall,potential,value_eur,wage_eur,player_positions,preferred_foot,international_reputation,weak_foot,skill_moves,work_rate,body_type,real_face,release_clause_eur,player_tags,team_position,team_jersey_number,loaned_from,joined,contract_valid_until,nation_position,nation_jersey_number,pace,shooting,passing,dribbling,defending,physic,gk_diving,gk_handling,gk_kicking,gk_reflexes,gk_speed,gk_positioning,player_traits,attacking_crossing,attacking_finishing,attacking_heading_accuracy,attacking_short_passing,attacking_volleys,skill_dribbling,skill_curve,skill_fk_accuracy,skill_long_passing,skill_ball_control,movement_acceleration,movement_sprint_speed,movement_agility,movement_reactions,movement_balance,power_shot_power,power_jumping,power_stamina,power_strength,power_long_shots,mentality_aggression,mentality_interceptions,mentality_positioning,mentality_vision,mentality_penalties,mentality_composure,defe

Como podemos ver, los elementos del RDD son solo los renglones del archivo CSV sin formato. Siendo el primer elemento los nombres de los campos de la base de datos, el mantener a los nombres de los campos como valores dentro del RDD puede derivar en problemas más adelante, así que por el momento los removeremos para quedarnos solamente con los valores

In [7]:
# Guarda el primer elemento 
columns = raw_data.take(1)[0]
# Guarda en todos los elementos de datos_crudos menos el primero correspondiente a las columnas
raw_data = raw_data.filter(lambda row : row != columns)

In [8]:
columns

'sofifa_id,player_url,short_name,long_name,age,dob,height_cm,weight_kg,nationality,club,overall,potential,value_eur,wage_eur,player_positions,preferred_foot,international_reputation,weak_foot,skill_moves,work_rate,body_type,real_face,release_clause_eur,player_tags,team_position,team_jersey_number,loaned_from,joined,contract_valid_until,nation_position,nation_jersey_number,pace,shooting,passing,dribbling,defending,physic,gk_diving,gk_handling,gk_kicking,gk_reflexes,gk_speed,gk_positioning,player_traits,attacking_crossing,attacking_finishing,attacking_heading_accuracy,attacking_short_passing,attacking_volleys,skill_dribbling,skill_curve,skill_fk_accuracy,skill_long_passing,skill_ball_control,movement_acceleration,movement_sprint_speed,movement_agility,movement_reactions,movement_balance,power_shot_power,power_jumping,power_stamina,power_strength,power_long_shots,mentality_aggression,mentality_interceptions,mentality_positioning,mentality_vision,mentality_penalties,mentality_composure,def

In [9]:
raw_data

PythonRDD[4] at RDD at PythonRDD.scala:53

Ahora podemos darle un poco más de estructura a nuestros datos. Primero separemos los campos de cada renglon, sabiendo que estan separados por comas, y quedandonos con solo un subconjunto de dichos campos, digamos... su nombre, edad, estatura, peso, nacionalidad, equipo y su ranking. Para esto definamos la siguiente función

In [10]:
def separar_campos(row):
    separated_row = row.split(',')
    # Extrae los campos que queremos
    nombre = str(separated_row[2])
    edad = int(separated_row[4])
    estatura = int(separated_row[6])
    peso = int(separated_row[7])
    nacionalidad = str(separated_row[8])
    equipo = str(separated_row[9])
    rank = int(separated_row[10])

    return (nombre, edad, estatura, peso, nacionalidad, equipo, rank)



aplicamos dicha función a cada renglon



In [11]:
rdd_fifa = raw_data.map(separar_campos)

# Imprimimos una muestra de los datos
for row in rdd_fifa.take(5):
    print(row)
print('\n')

('L. Messi', 32, 170, 72, 'Argentina', 'FC Barcelona', 94)
('Cristiano Ronaldo', 34, 187, 83, 'Portugal', 'Juventus', 93)
('Neymar Jr', 27, 175, 68, 'Brazil', 'Paris Saint-Germain', 92)
('J. Oblak', 26, 188, 87, 'Slovenia', 'Atlético Madrid', 91)
('E. Hazard', 28, 175, 74, 'Belgium', 'Real Madrid', 91)




Hasta ahora, hemos realizado diferentes operaciones sobre nuestro RDD para poder darle un formato más tratable a los datos que contiene. Sin embargo, no hemos detallado nada sobre la naturaleza de dichas operaciones. Enfoquemonos ahora en hacer más explícitos estos detalles.

Existen dos operaciones básicas que pueden ser aplicadas a los RDD:

    Acciones: Son operaciones que accionan la realización de cálculos y regresan valores explícitos del RDD.

    Transformaciones: Son operaciones que devuelven otro RDD. Este tipo de operaciones no se ejecutan como tal sino hasta que es llamada una acción. Por esta razón se dice que las transformaciones son de evaluacion peresoza (lazy evaluation).

En el caso de la celda anterior, map() es una transformación que aplica la función dada a los elementos del RDD y take() es la acción que nos devuelve los

primeros elementos.

Veamos más ejemplos. Podríamos querer saber cuantos jugadores existen por país en la base de datos. Para lograr esto primero necesitamos convertir nuestros datos en pares (key, value), donde la keyserá el pais y el value nos indicará que ese registro pertenece a un jugador de dicho país

In [12]:
# Nos quedamos solo con la entrada corresponiente al país
rdd_nacionalidades = rdd_fifa.map(lambda row: (row[4], 1))

Usando los países como llaves y la transfromación llamada reduceByKey, sumaremos todos los valores para cada país.

In [14]:
rdd_nacionalidades = rdd_nacionalidades.reduceByKey(lambda val_1 , val_2: val_1 + val_2)

In [15]:
rdd_nacionalidades

PythonRDD[10] at RDD at PythonRDD.scala:53

Hasta el momento hemos usado las transformaciones map() y reduceByKey(), para obtener resultados necesitamos efectuar una acción, en este caso usaremos la acción collect(), que nos devuelve todo el contenido del RDD. Esta acción debe usarse con cuidado, en casos donde el RDD contenga una gran cantidad de registros, cargar todo el contenido del RDD puede dejar al sistema sin memoria.

In [17]:
resultados = rdd_nacionalidades.collect()

for row in resultados[:20]:
    print(row)

('Argentina', 886)
('Portugal', 344)
('Brazil', 824)
('Slovenia', 61)
('Belgium', 268)
('Netherlands', 416)
('France', 984)
('Uruguay', 164)
('Poland', 324)
('Denmark', 345)
('Gabon', 16)
('Korea Republic', 322)
('Costa Rica', 30)
('Slovakia', 54)
('Bosnia Herzegovina', 66)
('Scotland', 277)
('Hungary', 35)
('Switzerland', 229)
('Greece', 96)
('Austria', 319)


Aunque los resultados obtenidos son buenos, sería interesante poder ordenar a los paises por la cantidad de jugadores. En la siguiente celda imprimimos los 10 países con más jugadores registrados en el juego de FIFA20

In [19]:
# Pedimos que ordene por total de jugadores
rdd_nacionalidades = rdd_nacionalidades.sortBy(lambda row: row[1], ascending = False)

print ("{:<12} {:<15}".format('PAIS', 'TOTAL JUGADORES'))

for row in rdd_nacionalidades.take(10):
    pais = row[0]
    total_jugadores = row[1]
  
    print ("{:<12} {:<15}".format(pais, total_jugadores))

PAIS         TOTAL JUGADORES
England      1667           
Germany      1216           
Spain        1035           
France       984            
Argentina    886            
Brazil       824            
Italy        732            
Colombia     591            
Japan        453            
Netherlands  416            


Además de las transformaciones y acciones empleadas aquí existen muchas más, sin embargo, los ejemplos expuestos aqui pueden ayudar al lector a familiarizarse con su uso.

Como el lector habrá notado, aunque los RDDs nos proveen de la capacidad de manipular y hacer calculos con datos distribuidos, éstos carecen de las funcionalidades y la estructura que tienen herramientas de un poco más alto nivel, como los DataFrame de Pandas, para el manejo de datos.

En PySpark existe un tipo de objetos, igual llamados DataFrames, los cuales nos ayudan a dotar a nuestros datos con más funcionalidades y estructura, en comparación a los RDDs, al mismo tiempo que seguir trabajando de forma distribuida con ellos. Por esta razon, procederemos a dar una introducción a los DataFrames de PySpark que son los entes con los cuales más trabajaremos a lo largo del curso.